# Pré-processamento — Water Potability

Neste notebook recebemos o dataset já limpo e preparamos os dados para a etapa de modelagem.
As duas transformações principais são o **escalonamento das features** e o **balanceamento das classes**.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Carregamento

In [5]:
df = pd.read_csv("dados/water_potability_limpo.csv")

print(f"Shape: {df.shape}")
print(f"\nDistribuição Potability:")
print(df["Potability"].value_counts())

Shape: (3276, 10)

Distribuição Potability:
Potability
0    1998
1    1278
Name: count, dtype: int64


## 2. Divisão Treino/Teste

Antes de qualquer transformação, separamos os dados em treino e teste.
Essa ordem é fundamental para evitar o **data leakage** — vazamento de informação
do conjunto de teste para o treino.

Se escalonássemos ou aplicássemos SMOTE **antes** da divisão, o modelo teria
acesso indireto à distribuição dos dados de teste durante o treino, resultando
em métricas de avaliação otimistas e não confiáveis.

Usamos **80% para treino e 20% para teste**, com `stratify=y` para garantir que
a proporção entre as classes seja mantida em ambos os conjuntos.

In [6]:
X = df.drop(columns=["Potability"])
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino : {X_train.shape[0]} amostras")
print(f"Teste  : {X_test.shape[0]} amostras")
print(f"\nDistribuição treino:")
print(y_train.value_counts())
print(f"\nDistribuição teste:")
print(y_test.value_counts())

Treino : 2620 amostras
Teste  : 656 amostras

Distribuição treino:
Potability
0    1598
1    1022
Name: count, dtype: int64

Distribuição teste:
Potability
0    400
1    256
Name: count, dtype: int64


## 3. Escalonamento — StandardScaler

### Por que escalonar?
As features do dataset estão em escalas muito diferentes — `Solids` chega a
dezenas de milhares (ppm) enquanto `ph` vai de 0 a 14. Algoritmos que calculam
distâncias ou gradientes (como SVM e Regressão Logística) são muito sensíveis a
isso: features com valores maiores dominam o cálculo e distorcem o aprendizado.

### Por que StandardScaler?
O `StandardScaler` transforma cada feature para ter **média 0 e desvio padrão 1**:

z = (x - média) / desvio padrão

Isso coloca todas as features na mesma escala sem distorcer a distribuição,
ao contrário do MinMaxScaler que comprime tudo entre [0,1] e é mais sensível a outliers.

### Fit apenas no treino
O scaler é ajustado (`fit`) **somente nos dados de treino** e depois aplicado (`transform`)
tanto no treino quanto no teste. Isso garante que a média e o desvio padrão usados
na transformação venham apenas do treino, sem contaminação do teste.

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print("Médias após escalonamento (treino):")
print(X_train_scaled.mean().round(4))
print("\nDesvios padrão após escalonamento (treino):")
print(X_train_scaled.std().round(4))

Médias após escalonamento (treino):
ph                 0.0
Hardness           0.0
Solids            -0.0
Chloramines       -0.0
Sulfate           -0.0
Conductivity      -0.0
Organic_carbon     0.0
Trihalomethanes    0.0
Turbidity          0.0
dtype: float64

Desvios padrão após escalonamento (treino):
ph                 1.0002
Hardness           1.0002
Solids             1.0002
Chloramines        1.0002
Sulfate            1.0002
Conductivity       1.0002
Organic_carbon     1.0002
Trihalomethanes    1.0002
Turbidity          1.0002
dtype: float64


## 4. Balanceamento — SMOTE

### O problema do desbalanceamento
O dataset tem ~60% de amostras não potáveis e ~40% potáveis. Embora não seja
um desbalanceamento extremo, modelos treinados nessa proporção tendem a favorecer
a classe majoritária — classificando tudo como "não potável" e ainda assim obtendo
boa acurácia, mas falhando justamente nos casos que mais importam.

### O que é o SMOTE?
O **SMOTE** (*Synthetic Minority Oversampling Technique*) gera amostras sintéticas
da classe minoritária ao invés de simplesmente duplicar linhas existentes.

O algoritmo funciona assim:
1. Para cada amostra da classe minoritária, encontra seus **k vizinhos mais próximos**
2. Cria novos pontos **ao longo do segmento** entre a amostra e um de seus vizinhos
3. Repete até equilibrar as classes

Isso é mais robusto do que duplicar amostras (oversampling simples), pois aumenta
a variabilidade dos dados sintéticos e reduz o risco de overfitting.

### SMOTE apenas no treino
O SMOTE é aplicado **somente no conjunto de treino**. Aplicá-lo antes da divisão
ou no conjunto de teste seria uma forma de data leakage — o modelo seria avaliado
em dados sintéticos, não em amostras reais.

In [8]:
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f"Antes do SMOTE  → classe 0: {(y_train == 0).sum()}  |  classe 1: {(y_train == 1).sum()}")
print(f"Depois do SMOTE → classe 0: {(y_train_bal == 0).sum()}  |  classe 1: {(y_train_bal == 1).sum()}")

Antes do SMOTE  → classe 0: 1598  |  classe 1: 1022
Depois do SMOTE → classe 0: 1598  |  classe 1: 1598


## 5. Exportação

Salvamos os quatro conjuntos separadamente para que o notebook de modelagem
possa carregá-los diretamente, sem precisar repetir nenhuma etapa de pré-processamento.

In [9]:
X_train_bal.to_csv("dados/X_train.csv", index=False)
X_test_scaled.to_csv("dados/X_test.csv", index=False)

pd.Series(y_train_bal, name="Potability").to_csv("dados/y_train.csv", index=False)
pd.Series(y_test,      name="Potability").to_csv("dados/y_test.csv",  index=False)

print("Arquivos salvos:")
print(f"  dados/X_train.csv  → {X_train_bal.shape}")
print(f"  dados/X_test.csv   → {X_test_scaled.shape}")
print(f"  dados/y_train.csv  → {y_train_bal.shape}")
print(f"  dados/y_test.csv   → {y_test.shape}")

Arquivos salvos:
  dados/X_train.csv  → (3196, 9)
  dados/X_test.csv   → (656, 9)
  dados/y_train.csv  → (3196,)
  dados/y_test.csv   → (656,)
